# Molecular Subtype Classification from a Consensus Differential-Expression Signature

**Supervised classification, external validation and unsupervised analysis of the TCGA-BRCA cohort restricted to a consensus set of differentially expressed genes**

## Overview

This notebook classifies breast tumours into their molecular subtype (Luminal A, Luminal B, Basal-like) using a reduced feature space: the consensus differential-expression signature (604 genes) obtained in the differential-expression step, instead of the full transcriptome.

The analysis proceeds in five stages:

1. **Data loading and label encoding** — the expression matrix is reshaped to a samples × genes layout and the subtype is parsed from the sample identifier.
2. **Train/test partition** — a stratified 70/30 split is drawn here and exported at patient level, so that every other modality of the project (histopathology, multi-omic integration) can reuse exactly the same partition.
3. **Supervised classification** — five classifiers (KNN, random forest, logistic regression, XGBoost, SVM) are tuned by stratified cross-validation and evaluated on the held-out test set, then compared by repeated stratified cross-validation.
4. **External validation** — the tuned models are applied to an independent Luminal B cohort (INC Bogotá, n = 23).
5. **Unsupervised analysis** — PCA, *k*-means, hierarchical clustering and UMAP on the same feature space.

> **This notebook is the origin of the train/test partition** used across the project (`split_transcriptomica_604.csv`). Re-running Section 2 with a different seed would invalidate the correspondence with the histopathology notebooks.

## Inputs and outputs

| File | Role |
|---|---|
| `Consensus_DEG_signature.tsv` | Input. Expression matrix of the signature genes, genes × samples, tab-separated. |
| `filtered_normalized_data_INC_23.csv` | Input. External INC Bogotá cohort, VST-normalised, semicolon-separated. |
| `split_transcriptomica_604.csv` | Output. Patient-level train/test partition (writing is disabled by default, see Section 2.3). |
| `top10_pc1_positive_genes.csv` | Output. Genes with the largest positive PC1 loadings, for enrichment analysis. |

## 1. Environment and data

### 1.1 Imports

All libraries used throughout the notebook are imported once here.

In [ ]:
# ============================================================
# Imports (consolidated)
# ============================================================
import os, sys, socket, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import randint, uniform
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

import sklearn
from sklearn.preprocessing import (LabelEncoder, LabelBinarizer,
                                   StandardScaler, MinMaxScaler)
from sklearn.impute import KNNImputer
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     RandomizedSearchCV, cross_val_score,
                                     StratifiedKFold, RepeatedStratifiedKFold, cross_validate)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, auc,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay,
                             silhouette_score, adjusted_rand_score)
from xgboost import XGBClassifier
import umap

warnings.filterwarnings('ignore')

### 1.2 File paths and constants

Paths and the class encoding are declared here so that the rest of the notebook contains no hard-coded values. Adjust the paths to the location of the data files.

In [ ]:
# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
SIGNATURE_FILE = "Consensus_DEG_signature.tsv"          # TCGA-BRCA, signature genes
INC_FILE       = "filtered_normalized_data_INC_23.csv"  # external cohort (INC Bogota)
SPLIT_FILE     = "split_transcriptomica_604.csv"        # partition exported in Section 2.3

RANDOM_STATE   = 42
TEST_SIZE      = 0.3

SUBTYPE_MAPPING = {'LumA': 0, 'LumB': 1, 'Basal': 2}
LABEL_MAP       = {0: 'LumA', 1: 'LumB', 2: 'Basal'}
LABEL_NAMES     = ['LumA', 'LumB', 'Basal']             # order 0, 1, 2
COLOR_MAP       = {'LumA': '#2196F3', 'LumB': '#FF9800', 'Basal': '#F44336'}

### 1.3 Environment record

Library versions and the interpreter in use are recorded for reproducibility.

In [ ]:
%pip list --format=freeze > requirements_rna.txt
!grep -iE "^(numpy|scipy|scikit-learn|xgboost|torch|timm|opencv|openslide)" requirements_rna.txt

In [ ]:
print("Python executable:", sys.executable)
print("NumPy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)

### 1.4 Loading the expression matrix

The file stores the signature genes as rows and the samples as columns. The first column (`Subtipo`) holds the row labels: the first row is the patient identifier (`IDPaciente`) and the remaining rows are genes. Sample columns follow the pattern `BRCA_<subtype>[.<n>]`, where the numeric suffix is added by pandas to disambiguate repeated column names.

In [ ]:
df = pd.read_csv(SIGNATURE_FILE, sep="\t")
print(f"Raw table: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()

### 1.5 Reshaping to a samples × genes matrix

The matrix is transposed so that each row is one sample. The subtype is extracted from the sample identifier with a regular expression.

In [ ]:
df_samples = df.set_index('Subtipo').T
df_samples.index.name = 'SampleID'
df_samples = df_samples.reset_index()
df_samples['Subtype'] = df_samples['SampleID'].str.extract(r'BRCA_([A-Za-z]+)', expand=False)

print(df_samples.head())
print(df_samples.shape)

### 1.6 Subtype label encoding

The categorical subtype labels are converted to integers with an explicit dictionary rather than `LabelEncoder`, so that the class order (LumA = 0, LumB = 1, Basal = 2) is fixed and identical in every model, table and figure of the project.

In [ ]:
df_samples['Subtype_Encoded'] = df_samples['Subtype'].map(SUBTYPE_MAPPING)

print(df_samples[['SampleID', 'Subtype', 'Subtype_Encoded']].head())
print("\nUnique subtypes and their encoded values:")
print(df_samples[['Subtype', 'Subtype_Encoded']].drop_duplicates().sort_values(by='Subtype_Encoded'))

## 2. Train/test partition

### 2.1 Stratified split

The samples are split into a training set (70 %) and a test set (30 %), stratified by subtype so that the class proportions are preserved in both sets. The features are every column of the matrix except the metadata columns.

In [ ]:
X = df_samples.drop(columns=['SampleID', 'IDPaciente', 'Subtype', 'Subtype_Encoded'])
y = df_samples['Subtype_Encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

### 2.2 Class distribution

Class sizes in the full cohort. The three subtypes are close to balanced, but macro-averaged metrics are used throughout the notebook so that the smallest class (Basal) is not under-weighted.

In [ ]:
df_samples['Subtype'].value_counts()

### 2.3 Exporting the partition at patient level

The partition is exported as a patient-level table so that the other modalities of the project can be trained and evaluated on exactly the same samples. Patient identifiers are truncated to the 12-character TCGA barcode (`TCGA-XX-XXXX`), which is the key shared with the histopathology tables.

The write is commented out on purpose: the file already exists and is consumed by the histopathology notebooks. Re-running the export after changing the seed or the data would break the correspondence between modalities.

In [ ]:
# =============================================
# EXPORT THE PATIENT IDs OF THE SPLIT
# =============================================

# The indices of X_train and X_test point to rows of df_samples
train_patient_ids = df_samples.loc[X_train.index, 'IDPaciente'].str[:12]
test_patient_ids  = df_samples.loc[X_test.index,  'IDPaciente'].str[:12]
train_subtypes    = df_samples.loc[X_train.index, 'Subtype']
test_subtypes     = df_samples.loc[X_test.index,  'Subtype']

df_train_ids = pd.DataFrame({
    'patient_id': train_patient_ids.values,
    'Subtype':    train_subtypes.values,
    'split':      'train'
})

df_test_ids = pd.DataFrame({
    'patient_id': test_patient_ids.values,
    'Subtype':    test_subtypes.values,
    'split':      'test'
})

df_split = pd.concat([df_train_ids, df_test_ids], ignore_index=True)

print("Split distribution:")
print(df_split['split'].value_counts())
print("\nSubtype distribution (train):")
print(df_split[df_split['split'] == 'train']['Subtype'].value_counts())
print("\nSubtype distribution (test):")
print(df_split[df_split['split'] == 'test']['Subtype'].value_counts())
print("\nExamples:")
print(df_split.head(13))

# Writing is disabled: the exported file is already consumed by the other modalities.
# df_split.to_csv(SPLIT_FILE, index=False)

## 3. Supervised models

### 3.1 Cross-validation strategy

Two cross-validation schemes are used and kept separate:

* `cv` — 5-fold stratified cross-validation on the training set, used exclusively for hyperparameter search.
* `cv_report` — 5-fold stratified cross-validation repeated three times (15 folds), used in Section 3.7 to compare the tuned models.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)                       # hyperparameter tuning
cv_report = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)         # final report

### 3.2 Model 1: k-nearest neighbours (KNN)

Standardisation and a KNN classifier are chained in a pipeline, so the scaler is re-fitted inside every cross-validation fold and no information leaks from the validation folds. The number of neighbours, the weighting scheme, the metric and the Minkowski exponent are tuned by exhaustive grid search (360 configurations × 5 folds).

In [ ]:
# k-nearest neighbours
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': range(1, 31),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['minkowski', 'cosine'],
    'knn__p': [1, 2, 3]
}

grid_clf = GridSearchCV(knn_pipeline,
                        param_grid=param_grid,
                        verbose=1,
                        return_train_score=True,
                        cv=cv,
                        scoring='accuracy')
grid_clf.fit(X_train, y_train)

The selected estimator and its hyperparameters:

In [ ]:
knn_best = grid_clf.best_estimator_
knn_best

In [ ]:
print("Best hyperparameters found by GridSearchCV:")
print(grid_clf.best_params_)

In [ ]:
# Cross-validated score of the best parameter combination
print(f"Best cross-validated accuracy: {grid_clf.best_score_:.4f}")

Evaluation on the held-out test set. The multi-class ROC AUC is computed one-vs-rest from the predicted class probabilities.

In [ ]:
# Predictions
y_pred = knn_best.predict(X_test)
# Multi-class ROC AUC requires the probabilities of every class
y_proba = knn_best.predict_proba(X_test)

# Metrics: 'average' set for the multi-class case
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Multi-class roc_auc_score: one-vs-rest with weighted averaging
auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print("Classification report:\n", classification_report(y_test, y_pred))

train_accuracy_knn = knn_best.score(X_train, y_train)
test_accuracy_knn = accuracy_score(y_test, y_pred)

print(f"Training accuracy (KNN): {train_accuracy_knn:.4f}")
print(f"Test accuracy (KNN): {test_accuracy_knn:.4f}")

In [ ]:
# Confusion matrix
label_names = LABEL_NAMES                 # 0, 1, 2
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix - KNN (test set)")
plt.show()

### 3.3 Model 2: Random forest

An ensemble of decision trees. Because the grid of a forest is large, the hyperparameters are explored by randomised search (50 sampled configurations × 5 folds). `max_features` is restricted to 2–18 % of the genes, which keeps the trees decorrelated in a high-dimensional feature space.

In [ ]:
# --- Base model ---
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

# --- Hyperparameter space for the randomised search ---
param_dist = {
    "n_estimators": [100, 200, 300, 500],              # number of trees
    "max_depth": [None, 30, 40, 50],                   # tree depth
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": [0.02, 0.05, 0.1, 0.15, 0.18],     # fraction of features per split (2-18%)
    "criterion": ["gini", "entropy"]
}

# --- Randomised search ---
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,                     # number of configurations evaluated
    scoring="accuracy",
    cv=cv,
    verbose=0,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# --- Fit the search ---
random_search.fit(X_train, y_train)

# --- Best hyperparameters ---
print("Best hyperparameters found:")
print(random_search.best_params_)

# --- Final evaluation ---
best_rf = random_search.best_estimator_

y_train_pred = best_rf.predict(X_train)
y_test_pred  = best_rf.predict(X_test)

print("\nClassification report:")
print(classification_report(y_test, y_test_pred))

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy  = accuracy_score(y_test,  y_test_pred)
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

In [ ]:
# Test-set predictions
y_test_pred = best_rf.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix - Random Forest (test set)")
plt.show()

### 3.4 Model 3: Logistic regression

A linear model with L1 or L2 penalty. The grid is written as a list of dictionaries because not every combination of penalty, solver and multi-class strategy is valid in scikit-learn: `liblinear` supports only one-vs-rest, `lbfgs` and `newton-cg` support only the L2 penalty, and `saga` supports both penalties. Splitting the grid this way evaluates only the valid combinations (40 in total).

In [ ]:
# Pipeline: standardisation + logistic regression
log_reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logisticregression', LogisticRegression(random_state=RANDOM_STATE))
])

# Parameter grid for GridSearchCV.
# Some combinations of penalty, solver and multi_class are incompatible;
# writing the grid as a list of dictionaries avoids them.
param_grid = [
    {
        'logisticregression__C': [0.001, 0.01, 0.1, 1],
        'logisticregression__penalty': ['l1', 'l2'],
        'logisticregression__solver': ['liblinear'],
        'logisticregression__multi_class': ['ovr']
    },
    {
        'logisticregression__C': [0.001, 0.01, 0.1, 1],
        'logisticregression__penalty': ['l2'],                      # lbfgs supports only l2 / none
        'logisticregression__solver': ['lbfgs', 'newton-cg'],
        'logisticregression__multi_class': ['ovr', 'multinomial']
    },
    {
        'logisticregression__C': [0.001, 0.01, 0.1, 1],
        'logisticregression__penalty': ['l1', 'l2'],                # saga supports l1 and l2
        'logisticregression__solver': ['saga'],
        'logisticregression__multi_class': ['ovr', 'multinomial']
    }
]

grid_search_log_reg = GridSearchCV(
    estimator=log_reg_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_log_reg.fit(X_train, y_train)

print("Best hyperparameters found: ", grid_search_log_reg.best_params_)
print("Best cross-validation accuracy: ", grid_search_log_reg.best_score_)

In [ ]:
# 1. Best model
best_log_reg_model = grid_search_log_reg.best_estimator_

# 2. Predictions on the test set
y_pred_log_reg = best_log_reg_model.predict(X_test)

# 3. Test accuracy
accuracy_log_reg = accuracy_score(y_test, y_pred_log_reg)

print("Best cross-validation accuracy: ", grid_search_log_reg.best_score_)
print(f"Test accuracy: {accuracy_log_reg:.4f}")

# 4. Classification report
class_report_log_reg = classification_report(y_test, y_pred_log_reg)
print("\nClassification report (logistic regression):\n", class_report_log_reg)

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_log_reg, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix - Logistic Regression (test set)")
plt.show()

### 3.5 Model 4: XGBoost

Gradient-boosted trees. Two elements differ from the previous models: univariate feature selection (`SelectKBest`) is placed before the classifier inside the pipeline, and class-balanced sample weights — inversely proportional to class frequency — are passed to the fit. Sixty configurations are sampled and scored by macro-averaged F1.

Note that `scipy.stats.uniform(loc, scale)` samples from the interval [`loc`, `loc + scale`], so the distributions above are read as ranges, not as mean and spread.

In [ ]:
# ------------------------------------------------------------------
# 1) Class weights (y is already encoded 0/1/2 = Subtype_Encoded)
# ------------------------------------------------------------------
label_map = LABEL_MAP                                   # same mapping across the whole pipeline

cls_counts    = pd.Series(y_train).value_counts()
class_weight  = (cls_counts.sum() / (len(cls_counts) * cls_counts)).to_dict()
sample_weight = pd.Series(y_train).map(class_weight).values

# ------------------------------------------------------------------
# 2) Pipeline: SelectKBest -> XGBClassifier
# ------------------------------------------------------------------
pipe_xgb = Pipeline(steps=[
    ("sel", SelectKBest(score_func=f_classif)),
    ("clf", XGBClassifier(
        objective="multi:softprob",
        num_class=len(np.unique(y_train)),
        tree_method="hist",
        n_jobs=-1,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE
    ))
])

# ------------------------------------------------------------------
# 3) Hyperparameter search (RandomizedSearchCV)
# ------------------------------------------------------------------
param_dist = {
    "sel__k":               [100, 200, 400, "all"],
    "clf__n_estimators":    randint(200, 600),
    "clf__max_depth":       randint(3, 8),
    "clf__learning_rate":   uniform(0.05, 0.3),
    "clf__subsample":       uniform(0.6, 0.3),
    "clf__colsample_bytree":uniform(0.4, 0.8),
    "clf__min_child_weight":randint(1, 5),
    "clf__reg_lambda":      uniform(0.5, 2.0),
    "clf__reg_alpha":       uniform(0.0, 1.0),
}

rs_xgb = RandomizedSearchCV(
    pipe_xgb,
    param_distributions=param_dist,
    n_iter=60,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
rs_xgb.fit(X_train, y_train, clf__sample_weight=sample_weight)
print("Best params (XGB):", rs_xgb.best_params_)
print("Best CV (f1_macro):", round(rs_xgb.best_score_, 3))
best_xgb = rs_xgb.best_estimator_

# -------------------
# TRAIN
# -------------------
y_train_pred = best_xgb.predict(X_train)
print("\nTraining accuracy (XGBoost):", round(accuracy_score(y_train, y_train_pred), 3))

# -------------------
# TEST
# -------------------
y_test_pred = best_xgb.predict(X_test)
print("Test accuracy (XGBoost):", round(accuracy_score(y_test, y_test_pred), 3))

# -------------------
# Confusion matrix
# -------------------
cm_xgb = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_xgb,
    display_labels=[label_map[i] for i in sorted(label_map)]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix - XGBoost (test set)")
plt.show()

In [ ]:
# TEST
y_test_pred = best_xgb.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
print("Test accuracy (XGBoost):", round(test_acc, 3))

print("\nClassification report - TEST (XGBoost):\n")
print(classification_report(
    y_test, y_test_pred,
    labels=[0, 1, 2],
    target_names=[label_map[i] for i in [0, 1, 2]]
))

### 3.6 Model 5: Support vector machine (SVM)

An SVM with a polynomial or radial-basis-function kernel, preceded by standardisation (SVMs are sensitive to feature scale). The full grid holds 540 configurations; `degree` and `coef0` affect only the polynomial kernel, so part of the grid is redundant for the RBF kernel.

In [ ]:
# Pipeline: standardisation + SVC
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=RANDOM_STATE))
])

param_grid_svm = {
    'svc__C': [0.01, 0.1, 1, 10, 100],
    'svc__kernel': ['poly', 'rbf'],
    'svc__gamma': ['scale', 1, 0.1, 0.01, 0.001, 0.0001],
    'svc__degree': [2, 3, 4],          # polynomial kernel only
    'svc__coef0': [0.0, 0.1, 1.0],     # polynomial kernel only
}

grid_search_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid_svm,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_svm.fit(X_train, y_train)

print("Best hyperparameters found: ", grid_search_svm.best_params_)
print("Best cross-validation accuracy: ", grid_search_svm.best_score_)

# Best estimator
best_svm_model = grid_search_svm.best_estimator_

# Predictions
y_pred_svm = best_svm_model.predict(X_test)

y_train_pred_svm = best_svm_model.predict(X_train)
train_accuracy_svm = accuracy_score(y_train, y_train_pred_svm)
print(f"Training accuracy (SVM): {train_accuracy_svm:.4f}")

y_test_pred_svm = best_svm_model.predict(X_test)
test_accuracy_svm = accuracy_score(y_test, y_test_pred_svm)
print(f"Test accuracy (SVM): {test_accuracy_svm:.4f}")

class_report_svm = classification_report(y_test, y_pred_svm)
print("\nClassification report (SVM):\n", class_report_svm)

In [ ]:
cm_svm = confusion_matrix(y_test, y_test_pred_svm, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=label_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix - SVM (test set)")
plt.show()

### 3.7 Model comparison by repeated cross-validation

A single test set of 166 samples gives a noisy estimate of performance, so the five tuned models are re-evaluated on the whole cohort with 5-fold stratified cross-validation repeated three times (15 folds per model). Macro-averaged metrics are reported because they weight the three subtypes equally.

Since every model sees the same folds, the per-fold scores are paired, and the two best-ranked models are compared with a paired Wilcoxon signed-rank test. The test answers whether their difference is systematic across folds or within fold-to-fold noise.

In [ ]:
from scipy.stats import wilcoxon

models = {"KNN": knn_best, "Random Forest": best_rf,
          "Logistic Regression": best_log_reg_model,
          "XGBoost": best_xgb, "SVM": best_svm_model}

# Macro metrics: robust to class imbalance
scoring = {
    "F1_macro":        "f1_macro",
    "Balanced_acc":    "balanced_accuracy",
    "Precision_macro": "precision_macro",
    "Recall_macro":    "recall_macro",
}

rows, f1_folds = [], {}
for name, model in models.items():
    res = cross_validate(model, X, y, cv=cv_report, scoring=scoring, n_jobs=-1)
    f1_folds[name] = res["test_F1_macro"]          # kept for the statistical test
    row = {"Model": name}
    for m in scoring:
        row[m] = f"{res['test_'+m].mean():.3f} +/- {res['test_'+m].std():.3f}"
    rows.append(row)

table = pd.DataFrame(rows).set_index("Model").sort_values("F1_macro", ascending=False)
print(table.to_string())

# --- Statistical test between the two best models ---
top2 = table.index[:2].tolist()
stat, p = wilcoxon(f1_folds[top2[0]], f1_folds[top2[1]])
print(f"\nWilcoxon {top2[0]} vs {top2[1]} (per-fold F1-macro): p = {p:.3f}")
print("Significant difference" if p < 0.05 else
      "No significant difference -> equivalent performance")

## 4. External validation: INC Bogotá cohort

The tuned models are applied to an independent cohort of 23 Luminal B samples with paired histopathology images. Because a single class is present, multi-class accuracy and F1 are not meaningful; the interpretable metric is the **sensitivity for Luminal B**, i.e. the fraction of samples assigned to the correct class.

Two caveats govern the interpretation of this section:

* The INC matrix is VST-normalised while TCGA is expressed as log2(FPKM + 1). No re-scaling or batch correction is applied, so part of any drop in performance reflects this difference in scale.
* Nine signature genes are absent from the INC panel and are imputed with a single constant (the global mean of the observed INC values), which removes their discriminative contribution.

### 4.1 Loading and gene-space alignment

The INC file is semicolon-separated and holds two metadata rows (`Response` and `slide_ID`) before the genes. The matrix is transposed and reindexed onto the gene space of the trained models.

In [ ]:
# 23 Luminal B samples with paired images. With a single class present, the
# interpretable metric is the sensitivity for Luminal B (fraction correctly
# classified); multi-class accuracy and F1 do not apply.

# -- Block 0: load, reshape and align to the gene space of the model --
raw = pd.read_csv(INC_FILE, sep=";", encoding="utf-8-sig")
raw = raw.rename(columns={raw.columns[0]: "rowlabel"}).set_index("rowlabel")
print("Index (first rows):", raw.index[:4].tolist())

# Metadata by POSITION: row 0 = Response (sample names), row 1 = slide_ID; genes from row 2
sample_names = raw.iloc[0].values
genes_inc    = raw.iloc[2:].copy()
genes_inc.columns = sample_names

df_inc = genes_inc.T.apply(pd.to_numeric, errors="coerce")
print("Shape of df_inc (samples x genes):", df_inc.shape)
print("First genes:", df_inc.columns[:3].tolist())

# Align to the gene space of the model -> builds X_inc
genes_model   = X_train.columns.tolist()
genes_missing = [g for g in genes_model if g not in df_inc.columns]
common_genes  = [g for g in genes_model if g in df_inc.columns]

col_means_inc = df_inc[common_genes].mean(axis=0)
X_inc = df_inc.reindex(columns=genes_model)
X_inc[genes_missing] = col_means_inc.mean()     # impute absent genes with the global INC mean
y_inc_true = np.ones(len(X_inc), dtype=int)

# Checks
print(f"Remaining NaN: {X_inc.isna().sum().sum()}")        # must be 0
print(f"Imputed missing genes ({len(genes_missing)}): {genes_missing}")
print(f"\nINC samples: {len(X_inc)} | Aligned genes: {X_inc.shape[1]}")

### 4.2 Expression distribution of the INC cohort

Global distribution of the expression values, as a histogram and as a kernel density estimate. This describes the scale of the external data before any prediction is made.

In [ ]:
vals = X_inc.values.flatten()
vals = vals[~np.isnan(vals)]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(vals, bins=50, color="steelblue", edgecolor="black", linewidth=0.3)
ax[0].set_title("Global distribution - INC"); ax[0].set_xlabel("Expression (VST)"); ax[0].set_ylabel("Frequency")

kde = stats.gaussian_kde(vals)
xr  = np.linspace(vals.min(), vals.max(), 300)
ax[1].fill_between(xr, kde(xr), alpha=0.3, color="steelblue")
ax[1].plot(xr, kde(xr), color="steelblue", lw=2)
ax[1].set_title("Density - INC"); ax[1].set_xlabel("Expression (VST)"); ax[1].set_ylabel("Density")
plt.tight_layout(); plt.show()

### 4.3 Comparability with TCGA

The INC samples are compared with the TCGA Luminal B samples only, since that is the class they belong to. Two views are shown: the per-sample median expression (overall scale) and the per-gene variance of both cohorts against the identity line (whether the same genes vary in both). Systematic departures here are expected given the different normalisation, and they bound how much can be concluded from the predictions that follow.

In [ ]:
X_lumb_tcga = (df_samples.loc[df_samples["Subtype"] == "LumB", genes_model]
               .apply(pd.to_numeric, errors="coerce"))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
# Per-sample medians
ax[0].hist(X_lumb_tcga.median(axis=1), bins=30, alpha=0.6, color="orange",
           label="TCGA LumB", edgecolor="black", linewidth=0.3)
ax[0].hist(X_inc.median(axis=1), bins=12, alpha=0.6, color="steelblue",
           label="INC LumB", edgecolor="black", linewidth=0.3)
ax[0].set_title("Per-sample median expression"); ax[0].set_xlabel("Median"); ax[0].legend()

# Per-gene variance
vt, vi = X_lumb_tcga.var(axis=0), X_inc.var(axis=0)
mx = max(vt.max(), vi.max())
ax[1].scatter(vt, vi, alpha=0.3, s=10, color="steelblue")
ax[1].plot([0, mx], [0, mx], "r--", alpha=0.5, label="identity")
ax[1].set_title(f"Per-gene variance ({X_inc.shape[1]} genes)")
ax[1].set_xlabel("TCGA LumB"); ax[1].set_ylabel("INC LumB"); ax[1].legend()
plt.tight_layout(); plt.show()

print(f"INC - mean: {X_inc.values.mean():.3f} | median: {np.median(vals):.3f} | "
      f"std: {X_inc.values.std():.3f} | range: [{X_inc.values.min():.2f}, {X_inc.values.max():.2f}]")

### 4.4 Sensitivity for Luminal B

Each model predicts the subtype of the 23 samples; the reported figure is how many are correctly assigned to Luminal B. Integer predictions are decoded to text through the shared label mapping, so that a different internal class order in any model cannot silently mislabel the results.

In [ ]:
print("Subtype encoding:", label_map)

models = {"KNN": knn_best, "Random Forest": best_rf, "Logistic Regression": best_log_reg_model,
          "XGBoost": best_xgb, "SVM": best_svm_model}

def to_text(pred):
    """Decode predictions to subtype names; pass them through if already text."""
    pred = np.asarray(pred)
    if pred.dtype.kind in ("U", "S", "O"):
        return pred.astype(str)
    return np.array([label_map[int(p)] for p in pred])

preds_txt, rows = {}, []
for name, model in models.items():
    txt = to_text(model.predict(X_inc)).astype(str)
    preds_txt[name] = txt
    n_ok = int((txt == "LumB").sum())
    rows.append({"Model": name, "Correct LumB": n_ok,
                 "Total": len(txt), "LumB sensitivity": round(n_ok / len(txt), 3)})

table = pd.DataFrame(rows).set_index("Model")
print("\n===== Sensitivity for Luminal B - INC cohort =====")
print(table.to_string())

### 4.5 Distribution of the predicted subtypes

All 23 samples are Luminal B, so any bar outside that class is an error. The shape of the errors is informative: samples falling into Luminal A suggest a threshold effect between the two luminal classes, whereas samples falling into Basal-like point to a more serious scale or batch problem.

In [ ]:
dist = (pd.DataFrame({n: pd.Series(t).value_counts() for n, t in preds_txt.items()})
        .T.reindex(columns=LABEL_NAMES, fill_value=0).fillna(0).astype(int))

ax = dist.plot(kind="bar", stacked=True, figsize=(9, 5),
               color=[COLOR_MAP[c] for c in LABEL_NAMES], edgecolor="black", linewidth=0.3)
ax.set_title(f"Predictions on the INC cohort (n={len(X_inc)}, all Luminal B)")
ax.set_ylabel("Number of samples"); ax.set_xlabel("")
ax.legend(title="Predicted subtype", loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 5. Unsupervised analysis

The structure of the data is explored without using the subtype labels, by dimensionality reduction (PCA, UMAP) and clustering (*k*-means, hierarchical). Each clustering is assessed with two complementary metrics: the **silhouette coefficient**, which measures internal cohesion and separation, and the **Adjusted Rand Index (ARI)** against the true subtypes, which measures external agreement. Labels are never used to fit these models; they only colour the projections and enter the ARI.

### 5.1 Preparation

The matrix is standardised (zero mean, unit variance per gene), which PCA, distance-based clustering and UMAP all require.

In [ ]:
# Numeric matrix (patients x genes) and true labels
X_num = X.apply(pd.to_numeric, errors='coerce')
X_num = X_num.fillna(X_num.mean())

labels_true = df_samples['Subtype'].values          # subtype (text)
y_true      = df_samples['Subtype_Encoded'].values  # encoded subtype (0, 1, 2)

# Standardisation (mean 0, variance 1): required for PCA / clustering / UMAP
X_scaled = StandardScaler().fit_transform(X_num)

print(f'Matrix: {X_scaled.shape[0]} patients x {X_scaled.shape[1]} genes')

### 5.2 PCA

The explained-variance profile is inspected first: the variance carried by each of the first components and the cumulative curve, with the number of components needed to reach 90 %.

In [ ]:
# Variance explained per component and cumulative
pca_full  = PCA(random_state=RANDOM_STATE).fit(X_scaled)
var_ratio = pca_full.explained_variance_ratio_
cum_var   = var_ratio.cumsum()
n_90 = int(np.argmax(cum_var >= 0.90) + 1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(range(1, 21), var_ratio[:20]*100, 'o-')
ax[0].set_title('Variance per component'); ax[0].set_xlabel('PC'); ax[0].set_ylabel('% variance')
ax[1].plot(range(1, len(cum_var)+1), cum_var*100, '-')
ax[1].axhline(90, color='r', ls='--', lw=0.8); ax[1].axvline(n_90, color='r', ls='--', lw=0.8)
ax[1].set_title(f'Cumulative variance (90% at {n_90} PCs)')
ax[1].set_xlabel('Number of components'); ax[1].set_ylabel('% cumulative')
plt.tight_layout(); plt.show()

print(f'PC1: {var_ratio[0]*100:.2f}%  |  PC2: {var_ratio[1]*100:.2f}%  |  90% with {n_90} components')

Two-dimensional projection coloured by the true subtype. Because the feature space is a differential-expression signature selected precisely to separate these classes, a strong first component is expected here.

In [ ]:
# 2D projection coloured by the true subtype
pca2  = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca2.fit_transform(X_scaled)
v = pca2.explained_variance_ratio_ * 100

plt.figure(figsize=(8, 6))
for sub, c in COLOR_MAP.items():
    m = labels_true == sub
    plt.scatter(X_pca[m, 0], X_pca[m, 1], c=c, label=f'{sub} (n={m.sum()})', alpha=0.6, s=25)
plt.xlabel(f'PC1 ({v[0]:.1f}%)'); plt.ylabel(f'PC2 ({v[1]:.1f}%)')
plt.title('PCA - true subtypes'); plt.legend(); plt.grid(alpha=0.2)
plt.tight_layout(); plt.show()

The genes with the largest loadings on PC1 are the ones that drive the main axis of variation. They are exported as input for functional-enrichment analysis (DAVID).

In [ ]:
# Genes with the largest PC1 loadings: input for enrichment analysis (DAVID)
loadings = pd.DataFrame(pca_full.components_[:2].T, index=X_num.columns, columns=['PC1','PC2'])
top_pc1_pos = loadings['PC1'].sort_values(ascending=False).head(10)
top_pc1_neg = loadings['PC1'].sort_values().head(10)
print('Top 10 PC1 (+):'); print(top_pc1_pos)
print('\nTop 10 PC1 (-):'); print(top_pc1_neg)
top_pc1_pos.rename('PC1_loading').to_csv('top10_pc1_positive_genes.csv')

### 5.3 K-means

The number of clusters is examined with two criteria: the elbow of the inertia curve and the mean silhouette coefficient, for *k* = 2 … 7. The final model is fitted with *k* = 3 to match the number of subtypes, which makes the ARI directly interpretable.

In [ ]:
# Choice of k: elbow method (inertia) + silhouette
K_range = range(2, 8)
inertias, sil = [], []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(K_range, inertias, 'o-'); ax[0].set_title('Elbow method'); ax[0].set_xlabel('k'); ax[0].set_ylabel('Inertia')
ax[1].plot(K_range, sil, 'o-', color='crimson'); ax[1].set_title('Silhouette vs k'); ax[1].set_xlabel('k'); ax[1].set_ylabel('Silhouette')
plt.tight_layout(); plt.show()

# Final fit with k = 3 (three subtypes)
km_labels = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit_predict(X_scaled)

### 5.4 Hierarchical clustering

Agglomerative clustering with Ward linkage, which merges the pair of clusters that increases within-cluster variance the least. The dendrogram is truncated to the last 12 merges for readability, and the tree is then cut into three clusters.

In [ ]:
# Ward linkage + truncated dendrogram
Z = linkage(X_scaled, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode='lastp', p=12, leaf_rotation=90., show_contracted=True)
plt.title('Dendrogram (Ward, truncated)'); plt.xlabel('Samples'); plt.ylabel('Distance')
plt.tight_layout(); plt.show()

# Cut into 3 clusters
hc_labels = fcluster(Z, 3, criterion='maxclust')

### 5.5 UMAP

A non-linear projection that, unlike PCA, preserves local neighbourhood structure. *k*-means is then applied in the UMAP space so that this approach yields cluster labels comparable with the previous two.

In [ ]:
# 2D UMAP on the standardised data
reducer   = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=RANDOM_STATE)
embedding = reducer.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for sub, c in COLOR_MAP.items():
    m = labels_true == sub
    plt.scatter(embedding[m, 0], embedding[m, 1], c=c, label=f'{sub} (n={m.sum()})', alpha=0.6, s=25)
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2'); plt.title('UMAP - true subtypes')
plt.legend(); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()

# Clustering in the UMAP space (for comparable metrics)
umap_labels = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit_predict(embedding)

### 5.6 Comparison of the clustering solutions

The silhouette coefficient is computed in the standardised gene space for all three methods, so the values are comparable; the ARI is computed against the true subtypes. A low silhouette together with a high ARI means the clusters recover the biological classes even though they are not geometrically compact — a common situation in high-dimensional expression data.

In [ ]:
# Silhouette (internal) and ARI vs true subtypes (external) for each clustering
def evaluate(name, labels):
    return {'Method': name,
            'Silhouette': round(silhouette_score(X_scaled, labels), 3),
            'ARI_vs_subtype': round(adjusted_rand_score(y_true, labels), 3)}

metrics = pd.DataFrame([
    evaluate('K-Means',             km_labels),
    evaluate('Hierarchical (Ward)', hc_labels),
    evaluate('UMAP + K-Means',      umap_labels),
]).set_index('Method')
print(metrics)
print(f'\nPCA  - PC1+PC2 explain {var_ratio[0]*100 + var_ratio[1]*100:.1f}% of the variance')
print('UMAP - 2D projection (n_neighbors=30, min_dist=0.1)')

Side-by-side view of the same PCA projection, coloured first by true subtype and then by *k*-means cluster. Each cluster takes the colour of its majority subtype, which makes the disagreements visible as points of one colour sitting inside a region of another.

In [ ]:
# Comparison panel: true subtypes vs K-means clusters (mapped to the majority subtype)
cluster_map = (pd.DataFrame({'k': km_labels, 'sub': labels_true})
               .groupby('k')['sub'].agg(lambda s: s.value_counts().index[0]).to_dict())
ari_km = adjusted_rand_score(y_true, km_labels)
sil_km = silhouette_score(X_scaled, km_labels)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for sub, c in COLOR_MAP.items():
    m = labels_true == sub
    ax[0].scatter(X_pca[m,0], X_pca[m,1], c=c, label=sub, alpha=0.6, s=20)
ax[0].set_title(f'True subtypes\nPC1={v[0]:.1f}% | PC2={v[1]:.1f}%'); ax[0].legend()
for k in range(3):
    m = km_labels == k
    ax[1].scatter(X_pca[m,0], X_pca[m,1], c=COLOR_MAP[cluster_map[k]],
                  label=f'Cluster {k} -> {cluster_map[k]} (n={m.sum()})', alpha=0.6, s=20)
ax[1].set_title(f'K-Means clusters (k=3)\nARI={ari_km:.3f} | Silhouette={sil_km:.3f}'); ax[1].legend()
for a in ax: a.set_xlabel('PC1'); a.set_ylabel('PC2')
plt.tight_layout(); plt.show()

## 6. Notes and limitations

1. **Feature selection and the test set.** The 604 genes were selected by differential expression on the full cohort, before this notebook splits the data. The test set therefore contributed to choosing the features, and its performance estimate is optimistic. A fully unbiased estimate would require the selection to be repeated inside each training fold.
2. **Repeated cross-validation (Section 3.7).** The comparison is run on the whole cohort with hyperparameters tuned on the training partition, which slightly favours all five models. Folds also share training data, so the per-fold scores are not independent and the Wilcoxon *p*-value is approximate.
3. **Class weights are not propagated.** The sample weights used to tune XGBoost are not passed to `cross_validate`, so Section 3.7 refits XGBoost without class weighting.
4. **XGBoost hyperparameter range.** `uniform(0.4, 0.8)` samples `colsample_bytree` from [0.4, 1.2], and XGBoost rejects values above 1. Roughly a quarter of the sampled configurations therefore fail and are scored as NaN. Use `uniform(0.4, 0.6)` to keep the range valid; note that this changes the search and requires re-running the model.
5. **External validation.** A single class, a different normalisation (VST vs log2(FPKM + 1)) and nine constant-imputed genes make the reported sensitivity a robustness check under distribution shift, not an estimate of clinical accuracy.
6. **Unsupervised analysis.** Clustering uses all samples, training and test alike. This is not label leakage, since the labels are not used to fit the models, but the metrics are descriptive and are not comparable with the supervised test-set results.